# Install packages

In [ ]:
!pip install 'brioche >= 0.5.1'

# Authenticate with Google

In [ ]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

print('Authenticated')

# Run biomization

In [ ]:
from brioche import BiomePftList, TaxaPftList, PollenCounts, Biomization, PollenPercentages, StabilizedPollenSamples
import pandas as pd
from collections import Counter

### Configuration

# The spreadsheet that contains one or more worksheets of pollen counts

# Without ages
# pollen_count_url = 'https://docs.google.com/spreadsheets/d/1uN3bDIEO834apewZPI5giQf2uymEGtaMATKnVLaAsZE'
# index_col = 0

# With ages
pollen_count_url = 'https://docs.google.com/spreadsheets/d/1l60DipabeYPxRKkn6z6PXNXLJgMxnxZ0mYWdkCjfgQk'
index_col = [0, 1]
#bin_years = 1000 # Size of window, start of the window will be multiples of this e.g. if bin_years=10000 then the 10000 label includes 10000-19999
#bin_years = 2000
bin_years = 5000



# Bin the biome affinity scores and recalculate
def bin(x):
  depth, age = x
  return ((int(age) + bin_years // 2) // bin_years) * bin_years

# Name of the result document (a timestamp is added)
result_file_name = f'Result test pollen counts ({bin_years} bins)'

# Name of the trace document containing partial steps of the calculation (a timestamp is added)
trace_file_name = f'Trace test pollen counts ({bin_years} bins)'

# Name of the spreadsheet and worksheets that contains the taxa/pft/biome mapping
biome_url = 'https://docs.google.com/spreadsheets/d/18ExYuMRQVwwFT9QwDIHQULbog_JA3h1r7DTHQzhkCTQ'
taxa_pft_worksheet_name = 'Taxa=PFT'
biome_pft_worksheet_name = 'PFT=BIOME'

default_threshold = 0.5

### Load data

pollen_document = gc.open_by_url(pollen_count_url)
biome_document = gc.open_by_url(biome_url)

biome_pft_mapping = BiomePftList.read_google_sheet(biome_document.worksheet(biome_pft_worksheet_name))
taxa_pft_mapping = TaxaPftList.read_google_sheet(biome_document.worksheet(taxa_pft_worksheet_name))

sites = []
for sheet in pollen_document.worksheets():
  print(sheet.title)
  if sheet.title in ('MM036-Gahinga', 'MM037-Muhavura'):
    sites.append(PollenPercentages.read_google_sheet(sheet, index_col=index_col))
  else:
    sites.append(PollenCounts.read_google_sheet(sheet, index_col=index_col))

print('Data loaded from Google drive')


### Perform biome affinity analysis

biomization = Biomization(taxa_pft_mapping, biome_pft_mapping)

unmapped_taxas = biomization.get_unmapped_taxas(*sites)
if unmapped_taxas:
  print()
  print('WARNING: sample worksheets contain taxas that are not mapped to any biome')
  for t in unmapped_taxas:
    print(t)
  print()

percentages = [sample.get_percentages(decimals=2) for sample in sites]
stabilized = [perc.get_stabilized(default_threshold=default_threshold, decimals=2) for perc in percentages]
biomes = [biomization.get_biome_affinity(stab) for stab in stabilized]


binned_biomes = [biome.apply(lambda scores: scores.groupby(bin).mean()) for biome in biomes]

for biome in binned_biomes:
  print('Site:', biome.site)
  print(biome.biomes)
  print()

# Store the result

In [ ]:
import time

def add_sheet(doc, name, data_with_index):
  print(name)
  data = data_with_index.reset_index()
  sheet = doc.add_worksheet(name, *data.shape)
  sheet.update([data.columns.values.tolist()] + data.values.tolist())

timestamp = time.strftime(' %Y-%m-%d %H:%M', time.localtime())

result_document = gc.create(result_file_name + timestamp)

for result in biomes:
  add_sheet(result_document, f'{result.site} all biomes', result.biomes)
  add_sheet(result_document, f'{result.site} all scores', result.scores)

for result in binned_biomes:
  add_sheet(result_document, f'{result.site} binned biomes', result.biomes)
  add_sheet(result_document, f'{result.site} binned scores', result.scores)

# Remove the empty first worksheet that is created by default
result_document.del_worksheet(result_document.sheet1)

print('Saved results to', result_file_name + timestamp, 'at', result_document.url)



# Store the traces

In [ ]:
trace_document = gc.create(trace_file_name + timestamp)
add_sheet(trace_document, 'Taxa/Biome matrix', biomization.taxa_biome_matrix)

#if unmapped_taxas:
#  sheet = trace_document.add_worksheet('Unmapped taxas', len(unmapped_taxas), 1)
#  sheet.update([[taxa] for taxa in unmapped_taxas])

for perc in percentages:
  add_sheet(trace_document, f'{perc.site} percentages', perc.samples)

for stab in stabilized:
  add_sheet(trace_document, f'{stab.site} stabilized', stab.samples)


trace_document.del_worksheet(trace_document.sheet1)

print('Saved trace to', trace_file_name + timestamp, 'at', trace_document.url)
